In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.semi_supervised import LabelSpreading
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix

# --- 辅助函数：计算临床评价指标 ---
def get_clinical_metrics(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tp / (tp + fn), tn / (tn + fp)

# --- (1) 数据读取与标准化 ---
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data"
columns = ["age", "sex", "cp", "trestbps", "chol", "fbs", "restecg", 
           "thalach", "exang", "oldpeak", "slope", "ca", "thal", "target"]
df = pd.read_csv(url, names=columns, na_values='?')
df.dropna(inplace=True)
df["target"] = (df["target"] > 0).astype(int)

# 关键步骤：特征标准化 (图方法必须步骤)
scaler = StandardScaler()
X = pd.DataFrame(scaler.fit_transform(df.drop("target", axis=1)), columns=df.columns[:-1])
y = df["target"]

print("数据标准化完成，准备构建图结构...")

数据标准化完成，准备构建图结构...


In [2]:
# --- (2) 构建半监督数据集 ---
# 划分训练集与独立测试集 (70% Train, 30% Test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# 模拟标注缺失：随机保留 20% 的有标签数据
rng = np.random.RandomState(42)
random_unlabeled_points = rng.rand(len(y_train)) < 0.8  # 80% 的数据将被屏蔽

# 构建混合标签向量 (y_train_mixed)
# - 有标签样本：保持原值 (0 或 1)
# - 无标签样本：赋值为 -1 (LabelSpreading 的专用标记)
y_train_mixed = y_train.copy()
y_train_mixed[random_unlabeled_points] = -1

print(f"=== 半监督数据集构建完成 ===")
print(f"训练集总数: {len(X_train)}")
print(f"  - 有标签样本 (Labeled): {np.sum(y_train_mixed != -1)}")
print(f"  - 无标签样本 (Unlabeled): {np.sum(y_train_mixed == -1)} (标记为 -1)")

=== 半监督数据集构建完成 ===
训练集总数: 207
  - 有标签样本 (Labeled): 43
  - 无标签样本 (Unlabeled): 164 (标记为 -1)


In [3]:
# --- (3) 标签传播模型训练 ---
# alpha=0.2: 软钳位系数，允许 20% 的信息来自邻居传播，增强抗噪性
# n_neighbors=7: 构建图时每个节点连接最近的 7 个邻居
lp_model = LabelSpreading(kernel='knn', n_neighbors=7, alpha=0.2, max_iter=30)

print("\n开始构建图模型并传播标签...")
# 注意：fit 过程中，标签会从非 -1 节点流向 -1 节点
lp_model.fit(X_train, y_train_mixed)


开始构建图模型并传播标签...


LabelSpreading(kernel='knn')

In [6]:
# --- (4) 最终模型评估 ---
# 关键步骤：执行归纳式推理 (Inductive Inference)
# 此时，模型利用学到的图结构特征，对从未见过的测试集样本进行预测
y_pred_lp = lp_model.predict(X_test)
y_prob_lp = lp_model.predict_proba(X_test)[:, 1]

# 计算各维度临床评价指标
acc_lp = accuracy_score(y_test, y_pred_lp)
auc_lp = roc_auc_score(y_test, y_prob_lp)
sens_lp, spec_lp = get_clinical_metrics(y_test, y_pred_lp)

print("\n=== 标签传播模型 (Label Spreading) 最终表现 ===")
print(f"Accuracy:    {acc_lp:.4f}")
print(f"AUC:         {auc_lp:.4f}")
print(f"Sensitivity: {sens_lp:.4f} (灵敏度)")
print(f"Specificity: {spec_lp:.4f} (特异度)")


=== 标签传播模型 (Label Spreading) 最终表现 ===
Accuracy:    0.8556
AUC:         0.9211
Sensitivity: 0.7619 (灵敏度)
Specificity: 0.9375 (特异度)
